# Solutions — Pagination: Page Numbers, Cursors, Links, Watermarks

Exercise notebook: `07_apis/03_pagination.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Page-number pagination

**Working solution**

In [ ]:
def fetch_all_customers(session, base_url, page_size=50):
    records, page = [], 1
    while True:
        resp = session.get(f"{base_url}/customers", params={"page": page, "page_size": page_size}, timeout=10)
        resp.raise_for_status()
        body = resp.json()
        records.extend(body["data"])
        if page >= body["total_pages"]:
            return records
        page += 1

**Explanation**

Trust the server's `total_pages`, but also stop on an empty page in production (servers lie when data changes mid-scan).

**Complexity:** O(n / page_size) requests.

**Common mistakes**

- Off-by-one: stopping before the last page or requesting one extra empty page forever.

## Exercise 2 — 🟡 Medium · Cursor pagination as a generator

**Working solution**

In [ ]:
def iter_orders(session, base_url, limit=100):
    cursor = None
    while True:
        params = {"limit": limit}
        if cursor:
            params["cursor"] = cursor
        resp = session.get(f"{base_url}/orders", params=params, timeout=10)
        resp.raise_for_status()
        body = resp.json()
        yield from body["items"]
        cursor = body.get("next_cursor")
        if not cursor:
            return

**Explanation**

`yield from` streams a page's items; the next request happens only when the consumer asks for more. `itertools.islice(iter_orders(...), 10)` fetches one page, not all.

**Complexity:** O(n / limit) requests, O(limit) memory.

**Common mistakes**

- Collecting into a list inside the generator — defeats laziness.

## Exercise 3 — 🟡 Medium · Follow next links

**Working solution**

In [ ]:
def iter_products(session, url, limit=50):
    seen = set()
    params = {"limit": limit}
    while url:
        if url in seen:
            raise RuntimeError(f"pagination loop detected at {url}")
        seen.add(url)
        resp = session.get(url, params=params, timeout=10)
        resp.raise_for_status()
        body = resp.json()
        yield from body["results"]
        url, params = body.get("next"), None

**Explanation**

Link-based pagination (GitHub, many REST APIs) — the server builds the URL, you just follow it. A `seen` set protects against infinite loops from buggy servers.

**Complexity:** O(pages).

**Common mistakes**

- Re-adding `limit` params to a `next` URL that already has them (duplicated params).

## Exercise 4 — 🟠 Hard · Incremental pull with a watermark

**Working solution**

In [ ]:
def pull_events(session, base_url, since, limit=100):
    watermark, events, seen = since, [], set()
    while True:
        resp = session.get(f"{base_url}/events", params={"since": watermark, "limit": limit}, timeout=10)
        resp.raise_for_status()
        body = resp.json()
        batch = body["events"]
        if not batch:
            return events, watermark
        for e in batch:
            if e["event_id"] not in seen:
                seen.add(e["event_id"])
                events.append(e)
        watermark = max(watermark, body["max_updated_at"])

**Explanation**

The watermark only moves forward (`max`). The loop ends on an empty batch. Note the edge case: if more than `limit` events share the *same* `updated_at`, a strict
`>` filter skips some — real APIs pair timestamps with an id cursor to avoid that.

**Complexity:** O(new events / limit) requests.

**Common mistakes**

- Storing the watermark *before* the load succeeds → data loss on failure.
- Using local time for the watermark.

## Challenge — 🔴 Very Hard · One paginator, three strategies

**Working solution**

In [ ]:
def _pages(session, url, style, page_size):
    if style == "page":
        page = 1
        while True:
            resp = session.get(url, params={"page": page, "page_size": page_size}, timeout=10)
            resp.raise_for_status()
            body = resp.json()
            yield body
            if page >= body.get("total_pages", page) or not body.get("data"):
                return
            page += 1
    elif style == "cursor":
        cursor = None
        while True:
            params = {"limit": page_size, **({"cursor": cursor} if cursor else {})}
            resp = session.get(url, params=params, timeout=10)
            resp.raise_for_status()
            body = resp.json()
            yield body
            cursor = body.get("next_cursor")
            if not cursor:
                return
    else:  # link
        params = {"limit": page_size}
        while url:
            resp = session.get(url, params=params, timeout=10)
            resp.raise_for_status()
            body = resp.json()
            yield body
            url, params = body.get("next"), None


def paginate(session, url, style, items_key, page_size=100, max_pages=None):
    if style not in ("page", "cursor", "link"):
        raise ValueError(f"unknown pagination style {style!r}")

    def records():
        for n, body in enumerate(_pages(session, url, style, page_size), start=1):
            yield from body.get(items_key) or []
            if max_pages is not None and n >= max_pages:
                return
    return records()

**Explanation**

Separating "how to get pages" (`_pages`) from "what to do with records" keeps each piece simple. Wrapping the generator in a regular function makes argument
validation eager: a generator function's body (including its `raise`) doesn't run until the first `next()`.

**Complexity:** O(pages).

**Common mistakes**

- Putting validation inside a generator → the error appears far from the call site.
- No page cap → a bug in the API burns your rate limit.